In [1]:
# first I will download all liabraies and initilaized global variables
#!pip -q install transformers datasets accelerate

import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_cosine_schedule_with_warmup
)
from torch.utils.data import DataLoader
from torch.optim import AdamW

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cuda


In [2]:
# Here I am loading data and splitting the trainin and testing datset
DATA_DIR = "./jigsaw-toxic-comment-classification-challenge"

train_path = os.path.join(DATA_DIR, "train.csv.zip")
test_path = os.path.join(DATA_DIR, "test.csv.zip")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

train_df["any_toxic"] = (train_df[label_cols].sum(axis=1) > 0).astype(int)

train_df, valid_df = train_test_split(
    train_df,
    test_size=0.05,
    random_state=42,
    stratify=train_df["any_toxic"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape:", test_df.shape)


Train shape: (151592, 9)
Valid shape: (7979, 9)
Test shape: (153164, 2)


In [4]:
# Here I choose roberta as base model and tokenize text
model_name = "gaunernst/bert-mini-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_length = 320

def tokenize_function(examples):
    return tokenizer(
        examples["comment_text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

train_dataset = Dataset.from_pandas(train_df[["comment_text"] + label_cols], preserve_index=False)
valid_dataset = Dataset.from_pandas(valid_df[["comment_text"] + label_cols], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["comment_text"]], preserve_index=False)

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

def add_labels(batch):
    labels = np.stack([batch[c] for c in label_cols], axis=-1)
    batch["labels"] = labels.astype("float32")
    return batch

train_dataset = train_dataset.map(add_labels, batched=True)
valid_dataset = valid_dataset.map(add_labels, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
valid_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])

print(train_dataset)
print(valid_dataset)
print(test_dataset)



Map:   0%|          | 0/151592 [00:00<?, ? examples/s]

Map:   0%|          | 0/7979 [00:00<?, ? examples/s]

Map:   0%|          | 0/153164 [00:00<?, ? examples/s]

Map:   0%|          | 0/151592 [00:00<?, ? examples/s]

Map:   0%|          | 0/7979 [00:00<?, ? examples/s]

Dataset({
    features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 151592
})
Dataset({
    features: ['comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 7979
})
Dataset({
    features: ['comment_text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 153164
})


In [5]:
BATCH_TRAIN = 16
BATCH_EVAL = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_TRAIN, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_EVAL, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_EVAL, shuffle=False)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("Test batches:", len(test_loader))


Train batches: 9475
Valid batches: 250
Test batches: 4787


In [6]:
loss_fn = torch.nn.BCEWithLogitsLoss()

def evaluate(model):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits

            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    probs = 1 / (1 + np.exp(-all_logits))
    scores = []
    for i in range(all_labels.shape[1]):
        if len(np.unique(all_labels[:, i])) < 2:
            continue
        scores.append(roc_auc_score(all_labels[:, i], probs[:, i]))
    return float(np.mean(scores))


In [7]:
def train_one_seed(seed, num_epochs=1):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label_cols),
        problem_type="multi_label_classification"
    )
    model.gradient_checkpointing_enable()
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    num_training_steps = num_epochs * len(train_loader)
    num_warmup_steps = int(0.1 * num_training_steps)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_roc = 0.0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = loss_fn(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            if (step + 1) % 200 == 0:
                avg_loss = running_loss / 200
                print(f"Seed {seed} Epoch {epoch+1}/{num_epochs} Step {step+1}/{len(train_loader)} Loss {avg_loss:.4f}")
                running_loss = 0.0

        val_roc = evaluate(model)
        print(f"Seed {seed} Epoch {epoch+1} validation ROC-AUC: {val_roc:.5f}")
        if val_roc > best_roc:
            best_roc = val_roc

    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
            all_logits.append(logits.cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    np.save(f"logits_seed{seed}.npy", all_logits)
    print(f"Saved logits_seed{seed}.npy with best ROC-AUC {best_roc:.5f}")
    return all_logits


In [8]:
def train_one_seed(seed, num_epochs=1):
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label_cols),
        problem_type="multi_label_classification"
    )
    model.gradient_checkpointing_enable()
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

    num_training_steps = num_epochs * len(train_loader)
    num_warmup_steps = int(0.1 * num_training_steps)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    best_roc = 0.0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = loss_fn(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            if (step + 1) % 200 == 0:
                avg_loss = running_loss / 200
                print(f"Seed {seed} Epoch {epoch+1}/{num_epochs} Step {step+1}/{len(train_loader)} Loss {avg_loss:.4f}")
                running_loss = 0.0

        val_roc = evaluate(model)
        print(f"Seed {seed} Epoch {epoch+1} validation ROC-AUC: {val_roc:.5f}")
        if val_roc > best_roc:
            best_roc = val_roc

    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
            all_logits.append(logits.cpu().numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    np.save(f"logits_seed{seed}.npy", all_logits)
    print(f"Saved logits_seed{seed}.npy with best ROC-AUC {best_roc:.5f}")
    return all_logits


In [9]:
seeds = [1]

for seed in seeds:
    out_path = f"logits_seed{seed}.npy"
    if os.path.exists(out_path):
        print(f"Seed {seed}: {out_path} already exists, skipping.")
        continue

    print(f"Starting training for seed {seed}...")
    _ = train_one_seed(seed=seed, num_epochs=2)
    print(f"Finished seed {seed}.")


Starting training for seed 1...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at gaunernst/bert-mini-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\alexa\AppData\Local\Temp\ipykernel_24612\543390299.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\alexa\AppData\Local\Temp\ipykernel_24612\543390299.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Seed 1 Epoch 1/2 Step 200/9475 Loss 0.6194
Seed 1 Epoch 1/2 Step 400/9475 Loss 0.4669
Seed 1 Epoch 1/2 Step 600/9475 Loss 0.3533
Seed 1 Epoch 1/2 Step 800/9475 Loss 0.2723
Seed 1 Epoch 1/2 Step 1000/9475 Loss 0.2215
Seed 1 Epoch 1/2 Step 1200/9475 Loss 0.1704
Seed 1 Epoch 1/2 Step 1400/9475 Loss 0.1342
Seed 1 Epoch 1/2 Step 1600/9475 Loss 0.1125
Seed 1 Epoch 1/2 Step 1800/9475 Loss 0.0866
Seed 1 Epoch 1/2 Step 2000/9475 Loss 0.0803
Seed 1 Epoch 1/2 Step 2200/9475 Loss 0.0787
Seed 1 Epoch 1/2 Step 2400/9475 Loss 0.0669
Seed 1 Epoch 1/2 Step 2600/9475 Loss 0.0683
Seed 1 Epoch 1/2 Step 2800/9475 Loss 0.0613
Seed 1 Epoch 1/2 Step 3000/9475 Loss 0.0621
Seed 1 Epoch 1/2 Step 3200/9475 Loss 0.0584
Seed 1 Epoch 1/2 Step 3400/9475 Loss 0.0580
Seed 1 Epoch 1/2 Step 3600/9475 Loss 0.0545
Seed 1 Epoch 1/2 Step 3800/9475 Loss 0.0578
Seed 1 Epoch 1/2 Step 4000/9475 Loss 0.0568
Seed 1 Epoch 1/2 Step 4200/9475 Loss 0.0530
Seed 1 Epoch 1/2 Step 4400/9475 Loss 0.0527
Seed 1 Epoch 1/2 Step 4600/9475 Loss

C:\Users\alexa\AppData\Local\Temp\ipykernel_24612\123686798.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Seed 1 Epoch 1 validation ROC-AUC: 0.97509


...

C:\Users\alexa\AppData\Local\Temp\ipykernel_24612\543390299.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Saved logits_seed1.npy with best ROC-AUC 0.97813
Finished seed 1.
Starting training for seed 2...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at gaunernst/bert-mini-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\alexa\AppData\Local\Temp\ipykernel_24612\543390299.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
C:\Users\alexa\AppData\Local\Temp\ipykernel_24612\543390299.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Seed 2 Epoch 1/2 Step 200/9475 Loss 0.6579
Seed 2 Epoch 1/2 Step 400/9475 Loss 0.4901


KeyboardInterrupt: 